# Optional modern-model replication
Baseline and detection only. Run after the main LLaVA experiment, on spare GPU time. Qwen2.5-VL-7B-Instruct is pinned in configs/pope_qwen_replication.json; the official model metadata reports Apache-2.0. The adapter is stub-tested; actual T4 memory fit is not yet verified. Keep this run separate from LLaVA and do not pool the results.


In [ ]:
from pathlib import Path
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/counterprior')
RUN=ROOT/'qwen-quick'
COCO=ROOT/'coco_val2014'
repo=Path('/content/CounterPrior-modern')
if not repo.exists():
    subprocess.run(['git','clone','--branch','claude/site-2026-11','https://github.com/basavarajnaduvinamani/DA5410-Winter-Project.git',str(repo)],check=True)
os.chdir(repo)
os.environ.update(COUNTERPRIOR_RUN=str(RUN),COUNTERPRIOR_COCO=str(COCO),COUNTERPRIOR_PROFILE='quick',MPLBACKEND='Agg')
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
subprocess.run(['bash','scripts/colab_run.sh','setup'],check=True)


In [ ]:
subprocess.run(['.venv-colab/bin/python','-m','scripts.run_pope','--config','configs/pope_qwen_replication.json','--profile','quick','--stages','baseline,calibrate,report','--output',str(RUN),'--coco-dir',str(COCO)],check=True)


In [ ]:
import zipfile
bundle=ROOT/'send_back_qwen_quick.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file() and 'states' not in p.relative_to(RUN).parts:
            z.write(p,Path('run')/p.relative_to(RUN))
print(bundle)
